# Introduction to Quantum Computing with Qiskit

This notebook starts from zero. You do not need previous experience with quantum physics or linear algebra. You need only basic Python, such as variables, loops, and functions.

We will use a simulator, which is an ordinary program that calculates what a small ideal quantum computer would do. The final topic is a small demonstration of Shor's factoring algorithm for the number 15.

The work time for this notebook is 2-3 hours, so work through as much as you can, take the rest home, and ask whatever questions you may have! 

## What you will learn

| Section | Topic |
|:---:|:---:|
| 1 | Qubits, measurement, probability, and histograms |
| 2 | Gates that change one qubit |
| 3 | Gates that connect qubits |
| 4 | Entanglement |
| 5 | The Quantum Fourier Transform |
| 6 | A small, simulated Shor's algorithm example |

Please only edit code cells that explicitly ask you to make changes, such as exercise cells. Other code cells may be used by later parts of the notebook, so if you accidentally modify or delete one, you can restore it by copying the original cell from the GitHub version of this notebook.



## How to work through this notebook

Run the setup cells, then work through the notebook in order. Pause at exercise cells and complete the marked instructions before running them. Unfinished starters may raise errors or produce incomplete results. If you do not want to work through an exercise yourself, skip its starter cell and run the provided solution cell instead. The optional challenges require you to complete their TODOs before execution.


> 
> **A note before you begin:** This notebook is designed to be comprehensive and accessible even if you are completely new to quantum computing. Starting around **Section 5 (The Quantum Fourier Transform)**, the circuits become significantly more complex, so the explanations also become much more detailed. There is a **lot** of text from this point onward because new concepts are explained as they appear rather than assumed as prior knowledge.
>
> Reading and fully understanding every explanation is **optional** if your main goal is to follow along with the code and see how the algorithms work. However, it is recommended if you want to understand **why** the circuits work rather than simply how to build and run them.
>


## A tiny math primer

A **vector** is an ordered list of numbers. In this notebook, a one-qubit state is represented by two numbers: one for the possibility of measuring 0 and one for the possibility of measuring 1. These numbers are called **amplitudes**. An amplitude can be negative or complex, so it is not itself a probability. To obtain a probability, take its magnitude and square it. This is called the Born rule, a fundamental postulate of quantum mechanics. It is not derived from quantum theory's other basic rules; rather, its predictions are supported by experimental observations.

For example, the state $\frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$ has amplitude $1/\sqrt{2}$ for each outcome. Each probability is $(1/\sqrt{2})^2 = 1/2$. The probabilities of all possible outcomes must add to 1.

You will see $i$, the imaginary unit, in some gate definitions. It is defined by $i^2=-1$. You do not need to calculate with it by hand here. It lets quantum states carry phase information, which gates can later turn into observable probability changes. Phase is the angle associated with a quantum amplitude; differences in phase determine how amplitudes reinforce or cancel one another when they combine. For example, amplitudes $+1/\sqrt{2}$ and $-1/\sqrt{2}$ each give the same probability, $1/2$, but have opposite phases, so they cancel if added together.

You may also see phase written as $e^{i\theta}$, where $\theta$ is an angle. This is a compact way to describe a rotation in the complex-number plane: $e^{i0}=1$, $e^{i\pi/2}=i$, $e^{i\pi}=-1$, and $e^{i\pi/4}$ is a 45-degree rotation. Multiplying an amplitude by $e^{i\theta}$ changes its angle by $\theta$ without changing its magnitude, so its immediate measurement probability stays the same.

## Reading results: histograms

The bar graphs used throughout this notebook are called **histograms**. Each label along the bottom is a measurement outcome, such as `0`, `1`, or `101`. The height of its bar is the number of times that outcome appeared after repeating the circuit. A circuit is run many times because a single quantum measurement is random whenever more than one outcome has nonzero probability. In Qiskit, one repetition is called a **shot**.

Qiskit prints bit strings with the highest-indexed classical bit on the left. For a two-qubit circuit measured with `measure([0, 1], [0, 1])`, the displayed string `10` means classical bit 1 is 1 and classical bit 0 is 0. The comments in each example point this out when it matters.


## 0. Setup

Run the next cell once in a new notebook environment. It installs Qiskit, the Python toolkit used to describe quantum circuits, and Aer, the simulator used to run them.


In [ ]:
# Install the packages needed by this notebook. Run this once per environment.
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pylatexenc matplotlib==3.11.2

## Install the packages needed by this notebook. Run this once per environment.

In [ ]:
# Imports used throughout the notebook.
from qiskit import QuantumCircuit, transpile
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator()
    AER_AVAILABLE = True
except ImportError:
    # This fallback supports the ideal examples if Aer was not installed yet.
    from qiskit.providers.basic_provider import BasicSimulator
    simulator = BasicSimulator()
    AER_AVAILABLE = False
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from math import gcd, pi
from fractions import Fraction

print("All set! Qiskit is ready to go.")
if not AER_AVAILABLE:
    print("Aer was not found, so the ideal examples will use Qiskit's BasicSimulator.")


---
## 1. Qubits, measurement, and your first circuit

### Bits and qubits

A classical **bit** stores exactly one of two values: 0 or 1. A **qubit** also gives 0 or 1 when measured, but before measurement it can be in a **superposition**, a state that assigns amplitudes to both outcomes.

We use **Dirac notation**: $|0\rangle$ means the state that always measures as 0, and $|1\rangle$ means the state that always measures as 1. The labels 0 and 1 inside the kets are just names, chosen to match the classical bit value that state will output when measured, not something derived. A general one-qubit state is

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle.$$

Here $\alpha$ and $\beta$ are amplitudes. Measurement produces 0 with probability $|\alpha|^2$ and 1 with probability $|\beta|^2$. The notation $|z|$ means the magnitude of $z$. The rule $|\alpha|^2+|\beta|^2=1$ makes the total probability equal to 1. 

### Circuits and measurement

A quantum **circuit** is a sequence of instructions applied from left to right. It has quantum wires, called qubits, and classical wires, called classical bits. Measurement transfers one qubit's result into one classical bit and ends the quantum part of that qubit's story for this circuit.


In [ ]:
# Create a circuit with 1 qubit and 1 classical bit
qc = QuantumCircuit(1, 1)

# Measure qubit 0, store the result in classical bit 0
qc.measure(0, 0)

# Draw the circuit
qc.draw("mpl")

**How to read these circuit diagrams:**
- **q** is the qubit line, this is your 1 quantum bit, starting in state |0⟩.
- **c** is the classical bit line, this is where measurement results get stored as ordinary 0s and 1s.
- The box with the **meter/gauge symbol** (⌒/) is the **measurement gate**, it's the operation that reads the qubit's state and converts it into a classical bit.
- The **arrow** pointing from q down to c shows *where* the result goes: qubit 0's measurement result is written into classical bit 0.
- The **"1"** above the c line is the width of the classical register (how many classical bits exist total, here, just 1). The **"0"** below it is the specific bit index being written to, not the value output of the qubit. To see the actual measured value(s), you need to simulate the circuit over many attempts (shots, discussed soon) and plot the results, usually with a histogram (`plot_histogram`), since a single run only gives one outcome and quantum results are inherently probabilistic across many runs.
- The **double diagonal line (⫽)** on the c line is notation meaning "this is a classical wire carrying multiple bits" (even though here it's just 1), it's just how Qiskit distinguishes classical wires from quantum wires visually.

Every qubit begins in $|0⟩$ unless we change it. The next cell repeats a measurement 1,000 times. The histogram should have one bar, `0`, with a height of 1,000.


In [ ]:
# Run the circuit 1000 times on the simulator
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()

print("Results:", counts)
plot_histogram(counts)

**Running the circuit**
- **`transpile(qc, simulator)`** rewrites your circuit into the specific gate set and layout the chosen simulator (or real hardware) actually understands, without changing what the circuit does. Every circuit needs this step before it can run.
- **`simulator.run(compiled, shots=1000)`** executes the compiled circuit 1000 times independently. Each run is called a **shot**. You need many shots because a single measurement only gives you one random outcome, and quantum results are probabilistic, so repeating it many times lets you see the actual probability distribution.
- **`.result()`** collects the outcomes from all 1000 shots into a result object.
- **`.get_counts()`** tallies those outcomes into a dictionary, like `{'0': 507, '1': 493}`, showing how many times each bitstring was measured. This is what you'll typically feed into `plot_histogram()` to visualize.

As expected, every shot returned `0`. The histogram is useful because it shows the distribution of results at a glance. We will next use gates to change that distribution.


---
## 2. Single-qubit gates

A **gate** is an operation on a qubit. Quantum gates are the analog of classical gates that operate on classical bits. Think of it as one instruction in a circuit. Gates are reversible before measurement, meaning that their effect can be undone by another gate.

### X gate, also called NOT

The X gate exchanges the two basis states:

$$X|0\rangle=|1\rangle, \qquad X|1\rangle=|0\rangle.$$

It is the quantum counterpart of a classical NOT operation.


In [ ]:
# Apply an X gate, then measure
qc = QuantumCircuit(1, 1)
qc.x(0)          # Flip qubit 0
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
# Run the X-gate circuit. Every shot should measure 1.
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("Results:", counts)
plot_histogram(counts)

### Hadamard gate, H

The Hadamard gate changes how the two possible outcomes combine. Applied to a qubit that starts as 0 or 1, it creates an even split between measuring 0 and measuring 1.

Mathematically:

$$H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle), \qquad H|1\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle).$$

These two superposition states appear so often that they have their own names:

$$|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle), \qquad
|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle).$$

So we can write the Hadamard operation more simply as

$$H|0\rangle = |+\rangle, \qquad H|1\rangle = |-\rangle.$$

When the H gate acts on a + or - state, it brings it back to a 0 or 1 state, respectively (explained next). Both outputs have the same $1/\sqrt{2}$ size on each term, so $|amplitude|^2 = 1/2$ either way, and an even 50/50 split when measured. The only difference between the two is a sign on the $|1\rangle$ term, plus in the first case, minus in the second.

If a qubit is already in a superposition, H does not simply put it "in superposition again." It combines the parts already there. They can reinforce one outcome and cancel the other, so H can turn an equal superposition into a state that will measure as definitely 0 or definitely 1. For example, applying $H$ to $\frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$ gives back $|1\rangle$ exactly, the minus sign caused the $|0\rangle$ terms to cancel and the $|1\rangle$ terms to add up.

H itself does not collapse the state; measurement does. Instead, H changes the state so that the later measurement can have a different result.

In [ ]:
# Put a qubit in superposition and measure
qc = QuantumCircuit(1, 1)
qc.h(0)           # Apply Hadamard
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
# Run the H-gate circuit. The histogram should be close to 50% 0 and 50% 1. It won't be exact!
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("Results:", counts)
plot_histogram(counts)

### Phase gates: Z, S, and T

The Z, S, and T gates change **phase**, which is the angle of an amplitude in the complex-number plane. They leave the $|0\rangle$ amplitude unchanged and rotate the $|1\rangle$ amplitude without changing its measurement probability.

$$Z|1\rangle=-|1\rangle=e^{i\pi}|1\rangle \quad \text{(180-degree rotation)}$$

$$S|1\rangle=i|1\rangle=e^{i\pi/2}|1\rangle \quad \text{(90-degree rotation)}$$

$$T|1\rangle=e^{i\pi/4}|1\rangle \quad \text{(45-degree rotation)}$$

An immediate measurement may show the same histogram, but later gates can use interference to turn these phase differences into different measurement probabilities.

In [ ]:
# H creates equal amplitudes. Z changes the sign of the |1> amplitude.
# The final H lets those amplitudes interfere, so the phase change becomes measurable.
qc_z = QuantumCircuit(1, 1)
qc_z.h(0)
qc_z.z(0)
qc_z.h(0)
qc_z.measure(0, 0)
qc_z.draw("mpl")

In [ ]:
# Simulate the circuit.
compiled = transpile(qc_z, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("H, then Z, then H:", counts)
plot_histogram(counts)

This circuit applies H, then Z, then H again, and every measurement comes out `1`, with no randomness. Here's why.

The first `h(0)` puts the qubit into an equal mix of $|0\rangle$ and $|1\rangle$: $(|0\rangle+|1\rangle)/\sqrt{2}$. Measuring here would give 0 or 1 with 50/50 probability.

`z(0)` doesn't change those probabilities, it flips the sign on the $|1\rangle$ part only: $(|0\rangle-|1\rangle)/\sqrt{2}$. If you measured right now, you'd still get 50/50, that sign flip is invisible to a direct measurement.

The second `h(0)` is what makes the sign flip visible. Applying H again causes the $|0\rangle$ and $|1\rangle$ parts to combine, and because of that earlier sign flip, they combine into pure $|1\rangle$ instead of back into the original state. This combining effect, where amplitudes add up or cancel out depending on their signs, is called **interference**, and it's why the final measurement is guaranteed to be 1, not random at all.


### Exercise 1: predict, build, and check

For each circuit, predict whether the histogram will show only `0`, only `1`, or a roughly even split. Then run the starter cell. Every circuit must end with `measure(0, 0)` because a simulator cannot put a quantum result into a histogram until it is measured.

1. `X`, then `H`, then measurement, beginning in $|0\rangle$.
2. `H`, then `Z`, then `H`, then measurement, beginning in $|0\rangle$.
3. `H`, then `H`, then measurement, beginning in $|0\rangle$. Hint: $H$ is its own inverse.


In [ ]:
# Exercise 1 starter circuits. These are complete and runnable examples.
# Read the gate order, predict the result, then compare with the printed counts.
def run_one_qubit(circuit, label):
    compiled = transpile(circuit, simulator)
    counts = simulator.run(compiled, shots=1000).result().get_counts()
    print(label, counts)
    return counts

qc1 = QuantumCircuit(1, 1)
qc1.x(0)
qc1.h(0)
qc1.measure(0, 0)
run_one_qubit(qc1, "1. X then H:")

qc2 = QuantumCircuit(1, 1)
qc2.h(0)
qc2.z(0)
qc2.h(0)
qc2.measure(0, 0)
run_one_qubit(qc2, "2. H then Z then H:")

qc3 = QuantumCircuit(1, 1)
qc3.h(0)
qc3.h(0)
qc3.measure(0, 0)
run_one_qubit(qc3, "3. H then H:"); # Semicolon suppresses the duplicate auto-printed return value (housekeeping addition).


### Exercise 1 explanation

Circuit 1 ends in an equal superposition, so it gives a roughly 50/50 split between `0` and `1`. Circuit 2 always returns `1`: the first H creates $|+\rangle$, Z changes it to $|-\rangle$ by flipping the relative phase, and the final H transforms $|-\rangle$ into $|1\rangle$. Circuit 3 always returns `0`, because $HH=I$, where $I$ is the identity operation that changes nothing.

In [ ]:
# Exercise 1 solution visualized as three histograms.
for circuit, label in [(qc1, "X then H"), (qc2, "H then Z then H"), (qc3, "H then H")]:
    counts = run_one_qubit(circuit, label)
    fig, ax = plt.subplots()
    plot_histogram(counts, title=label, ax=ax)
    plt.show()

---
## 3. Gates that connect qubits (multi-qubit gates)

So far, each gate has acted on one qubit at a time. A **multi-qubit gate** acts on two or more qubits together. This lets us do things such as flip one qubit depending on another qubit's value, or exchange the states of two qubits. We will start with a conditional flip, then try the Toffoli and SWAP gates.

### CNOT gate, also called controlled-X or CX

Remember that the X gate flips $|0\rangle$ to $|1\rangle$ and $|1\rangle$ to $|0\rangle$. **CNOT** means **controlled-NOT**: it applies that same flip to one qubit, but only when a second qubit is in $|1\rangle$.

The two qubits have different jobs:

- The **control** determines whether the flip happens. If it is 0, leave the target alone; if it is 1, flip the target.
- The **target** is the qubit that receives the X operation when the control is 1.

For inputs that are definitely 0 or 1, the full rule is:

| Control before | Target before | Control after | Target after |
|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 1 | 1 |
| 1 | 1 | 1 | 0 |

Notice that the control's value stays the same. The target is **flipped**, not simply set to 1: if both inputs are 1, the target becomes 0. Applying the same CNOT twice undoes the flip and returns the original state.

In Qiskit, `qc.cx(0, 1)` makes qubit 0 the control and qubit 1 the target. The first argument is the control, and the second is the target. In the circuit drawing, a filled dot marks the control and a circled plus marks the target; a line joins them to show that they belong to the same gate.

### Walk through the next two examples

Every qubit starts in $|0\rangle$. In the first circuit below, the control is 0, so CNOT leaves the target at 0. Every shot should give `00`.

In the second circuit, `qc.x(0)` first changes the control to 1. CNOT then flips the target from 0 to 1, so every shot should give `11`.

Both circuits use `qc.measure([0, 1], [0, 1])`, which stores qubit 0's result in classical bit 0 and qubit 1's result in classical bit 1. Remember that Qiskit displays these results as `c1c0`, so the target's result is on the left and the control's result is on the right.

### CNOT and Entanglement Aside:

CNOT applies its flip rule directly to the quantum state, it never checks or collapses anything first. If the control qubit is in superposition, that superposition has multiple terms, and CNOT acts on each term separately, using its normal rule: flip the target if that term's control value is 1, leave it if that term's control value is 0.

For example, if $q_0$ is in superposition $(|0\rangle+|1\rangle)/\sqrt{2}$ and $q_1$ starts at $|0\rangle$, the joint state is $(|00\rangle+|10\rangle)/\sqrt{2}$. CNOT(control=$q_0$, target=$q_1$) leaves the $|00\rangle$ term alone (control is 0) and flips the $|10\rangle$ term to $|11\rangle$ (control is 1), giving $(|00\rangle+|11\rangle)/\sqrt{2}$. (This explanation uses normal `q0q1` ordering, not Qiskit's reverse ordering.)

Now $q_0$ and $q_1$ can't be described separately, measuring one instantly tells you the other. That correlation is **entanglement**, and you'll build it directly in Section 4.
For now, the two examples below use a control that's already definite (0 or 1), so you can check the conditional-flip rule in its simplest form before superposition enters the picture.

In [ ]:
# Two qubits (q0, q1) and two classical result slots (c0, c1).
# Start with q0 = 0, so this CNOT should not change q1.
qc = QuantumCircuit(2, 2)
# cx(control, target): q0 is the control and q1 is the target.
# q1 flips only when q0 is 1; q0 itself is never changed.
qc.cx(0, 1)
# measure([qubits], [classical bits]): each position pairs up, so q0's result goes to c0 and q1's result goes to c1.
# Qiskit prints results as c1c0, so the leftmost displayed bit is q1's result.
qc.measure([0, 1], [0, 1])
print("Control q0 = |0> (so target q1 stays |0>):")
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

As expected, q1 never flips since the control q0 was 0. Both bits report as 0 with certainty, no other outcome is possible here.

In [ ]:
# Now flip the control to |1> first, then CNOT
qc = QuantumCircuit(2, 2)
qc.x(0)            # Set control to |1>
qc.cx(0, 1)        # CNOT: now the target flips!
qc.measure([0, 1], [0, 1])
print("Control = |1>:")
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Now that `q0` is `1`, CNOT flips `q1` from `0` to `1`, producing `11`. This demonstrates a controlled flip. These qubits are not entangled: both have definite states. In Section 4, we will put the control into superposition and use CNOT to create entanglement.

### Toffoli gate, also called CCX

The Toffoli gate has two controls and one target. It flips the target only when both controls are 1. In Qiskit, `ccx(control_a, control_b, target)` applies it. It is useful because it provides conditional logic with more than one condition.


In [ ]:
# Toffoli: both controls must be |1> to flip the target
qc = QuantumCircuit(3, 3)
qc.x(0)              # Control 1 = |1>
qc.x(1)              # Control 2 = |1>
qc.ccx(0, 1, 2)      # Toffoli: flip qubit 2, so now due to the 2 X gates and the Toffoli, we get |111>, which we see in the histogram output in the next cell.
qc.measure([0, 1, 2], [0, 1, 2])
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
print("Both controls |1> -> target flips to |1>:")
plot_histogram(result.get_counts())

The Toffoli only flips its target when *both* controls are 1, so this is a stricter, two-condition version of the single-control CNOT you saw previously.

### SWAP gate

`swap(a, b)` exchanges the complete states of two qubits. For example, if qubit 0 is $|1\rangle$ and qubit 1 is $|0\rangle$, then

$$|01\rangle \rightarrow |10\rangle$$

using Qiskit's $|q_1q_0\rangle$ ordering. After the SWAP, qubit 0 is $|0\rangle$ and qubit 1 is $|1\rangle$.

It is not a measurement or a copy: it is a reversible exchange. If one qubit is in a superposition, such as $|+\rangle$, and the other is $|0\rangle$, then

$$|0\rangle|+\rangle \rightarrow |+\rangle|0\rangle.$$

The superposition simply moves to the other qubit. Nothing gets measured or collapsed, and in this case the qubits do not become entangled.

In [ ]:
# SWAP: exchange qubit 0 (|1>) and qubit 1 (|0>)
qc = QuantumCircuit(2, 2)
qc.x(0)              # qubit 0 = |1>, qubit 1 = |0>
qc.swap(0, 1)        # Now qubit 0 = |0>, qubit 1 = |1>. What do you think this would output in the histogram as? Remember qiskit's leftmost-first convention!
qc.measure([0, 1], [0, 1])
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Before the swap, q0 held `1` and q1 held `0`. After `swap(0, 1)`, those values trade places, so q0 becomes `0` and q1 becomes `1`. Reading the histogram as c1c0 (Qiskit's leftmost-first convention), `10` means c1=1 and c0=0, exactly the swapped result, with no randomness since no superposition was involved.

### Exercise 2: build a quantum half-adder

A half-adder takes two input bits and outputs their **sum** and **carry**, exactly like adding by hand: 1+1 = 0, carry 1.

Addition without carrying is just XOR. No gate is named "XOR," so your job is to find which gate (or combination) *behaves* like XOR for the sum, and which behaves like the carry.

XOR Review:

| bit A | bit B | A XOR B |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

You have X, CNOT, and Toffoli. Combine them so qubit 2 ends up holding the sum, and qubit 3 ends up holding the carry, given inputs on qubits 0 and 1.

Test all four input combinations below and check against binary addition: 0+0=0 carry 0, 0+1=1 carry 0, 1+0=1 carry 0, 1+1=0 carry 1. This code is given to you.

Once one input pair works, don't copy-paste four circuits. Wrap it in a function that takes `bit0` and `bit1`, using `if` to decide whether to add the `x` gates, then loop over the four pairs. This function has been started for you.

In [ ]:
# Exercise 2: build a quantum half-adder.
# q0, q1 = the two input bits. q2 = sum output. q3 = carry output.
# Don't hardcode four separate circuits, write one function that takes
# bit0 and bit1 as arguments and builds the circuit for those inputs.

def half_adder(bit0, bit1):
    qc = QuantumCircuit(4, 4)

    # Set the two input qubits based on bit0 and bit1.
    if bit0 == 1:
        # ...
    if bit1 == 1:
        # ...

    # TODO: add the gates that make q2 hold the sum and q3 hold the carry.
    # You have qc.cx(control, target) and qc.ccx(control1, control2, target)
    # available. Think about how many gates you need for each output qubit.

    # ...

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])
    return qc

# Test all four input combinations.
for bit0, bit1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    circuit = half_adder(bit0, bit1)
    counts = simulator.run(transpile(circuit, simulator), shots=1000).result().get_counts()
    print(f"{bit0} + {bit1}:", counts)

### Exercise 2 explanation

The missing gates are (expand this section after attempting the exercise):

In [ ]:
qc.cx(0, 2)
qc.cx(1, 2)
qc.ccx(0, 1, 3) # Don't run this cell, run the next python cell to see the circuit. 

**Sum (qubit 2):** addition without carrying is XOR, and no gate is directly named XOR, but two CNOTs into the same target reproduce it. Each CNOT flips qubit 2 independently if its own control is 1. If only one input is 1, qubit 2 gets flipped once and lands on 1. If both inputs are 1, qubit 2 gets flipped twice, which cancels back to 0. If neither is 1, qubit 2 is never touched and stays 0. That matches the XOR table.

**Carry (qubit 3):** carry is 1 only when *both* inputs are 1, which is precisely what Toffoli already does by definition, flip the target only if both controls are 1. No combination needed here, just a direct application of the gate.

In [ ]:
# Exercise 2 solution.
def half_adder(bit0, bit1):
    qc = QuantumCircuit(4, 4)

    if bit0 == 1:
        qc.x(0)
    if bit1 == 1:
        qc.x(1)

    qc.cx(0, 2)
    qc.cx(1, 2)
    qc.ccx(0, 1, 3)

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])
    return qc


for bit0, bit1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    circuit = half_adder(bit0, bit1)

    # Display the input combination and its circuit.
    print(f"\n{bit0} + {bit1}")
    display(circuit.draw("mpl"))

    # Run the circuit.
    counts = simulator.run(
        transpile(circuit, simulator),
        shots=1000
    ).result().get_counts()

    print("Result:", counts)

### Reading the output

Qiskit displays the measured qubits from highest to lowest index, so the output bitstring is ordered as

$$
q_3q_2q_1q_0
$$

Here, $q_0$ and $q_1$ are the two input bits, $q_2$ stores the **sum**, and $q_3$ stores the **carry**.

Therefore, the four outputs read as:

- **0 + 0:** `0000` → carry = 0, sum = 0, inputs = 00 → result is **00**
- **0 + 1:** `0110` → carry = 0, sum = 1, inputs = 10 → result is **01**
- **1 + 0:** `0101` → carry = 0, sum = 1, inputs = 01 → result is **01**
- **1 + 1:** `1011` → carry = 1, sum = 0, inputs = 11 → result is **10**

---
## 4. Building circuits: entanglement

### The Bell state

**Entanglement** occurs when qubits become connected in a way that cannot be fully described by looking at each qubit separately. Their quantum state must instead be described as a joint state of the qubits. This can produce **correlations**, where measuring one qubit gives us information about the measurement of the other. **Correlation** means that the measurement outcomes are related: knowing the result of one can tell us something about the likely or certain result of the other. Entanglement does not allow information to be sent faster than light.

Why? Each person's measurement result is individually random. You cannot control whether your entangled qubit gives 0 or 1, so you cannot use it to encode a message. The correlation only becomes apparent after the two people compare their results using ordinary classical communication. In short, instantaneous correlation $\neq$ instantaneous communication.

Apply H to qubit 0, then CNOT from qubit 0 to qubit 1, using the qiskit $|q1q0\rangle$ convention:

$$|00\rangle \xrightarrow{H\text{ on }q_0} \frac{|00\rangle + |01\rangle}{\sqrt{2}} \xrightarrow{\mathrm{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

After H, qubit 0 is in an equal superposition. The CNOT acts on both parts of the superposition: it leaves the $q_0=0$ part unchanged and flips qubit 1 in the $q_0=1$ part, leaving the two possibilities $|00\rangle$ and $|11\rangle$.

Each individual qubit has a 50% chance of being measured as 0 or 1. However, their results are perfectly correlated: if one is measured as 0, the other will also be 0; if one is measured as 1, the other will also be 1. Therefore, we measure either 00 or 11, while 01 and 10 never appear on an ideal simulator.

### Why this is called a Bell state

The state we just created,

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}},$$

is one of the **Bell states**, a set of four two-qubit states that are maximally entangled.

What makes this state interesting is that neither qubit's measurement can be predicted on its own: each has a 50% chance of being measured as 0 or 1. However, the two results are connected. If one qubit is measured as 0, the other will also be measured as 0; if one is measured as 1, the other will also be measured as 1.

So although we cannot predict **which** result we will get, we can predict the **relationship between the two results**: they will always match.

The four Bell states are:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}, \qquad |\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}},$$

$$|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \qquad |\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}.$$

In [ ]:
# Create the Bell state Phi-plus
bell = QuantumCircuit(2, 2)
bell.h(0)             # Superposition on qubit 0
bell.cx(0, 1)         # Entangle qubit 1 with qubit 0
bell.measure([0, 1], [0, 1])

bell.draw("mpl")

In [ ]:
# Run it , only '00' and '11' should appear (never '01' or '10')
compiled = transpile(bell, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Bell states are useful in many important quantum computing and communication protocols. In **quantum teleportation**, a shared Bell pair helps transfer the quantum state of a qubit using entanglement and classical communication. In **superdense coding**, a shared Bell pair allows two classical bits of information to be communicated by sending only one qubit. Bell states are also used in **quantum cryptography**, where their measurement correlations can help establish secure communication, and in **quantum networks**, where Bell pairs can provide entanglement between distant quantum devices (such as quantum computers or processors). They also appear in **quantum error correction**, where entanglement and correlations between qubits are used to detect and correct errors, and in **Bell tests**, which experimentally demonstrate correlations that cannot be explained by local hidden-variable theories (theories where measurement outcomes are determined by pre-existing, locally stored information).

### Exercise 3: related entangled states

1. Build a two-qubit circuit that creates $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$. The only measurement results should be `01` and `10`. *(P.S. We recommend you then try creating the two other Bell states.)*
2. Build a three-qubit circuit that creates the GHZ (Greenberger-Horne-Zeilinger) state $(|000\rangle + |111\rangle)/\sqrt{2}$. The only measurement results should be `000` and `111`.

For each state, create the `QuantumCircuit`, add the gates needed to produce the state, and measure all qubits. The simulation code is provided below so you can run your circuits and check whether the measurement results match the expected outcomes.


In [ ]:
# Exercise 3.1: create the |Psi+> Bell state.
qc_bell2 = QuantumCircuit(2, 2)
# Add your gates here.
qc_bell2.measure([0, 1], [0, 1])

# Run Exercise 3.1.
counts_bell = simulator.run(
    transpile(qc_bell2, simulator),
    shots=1000
).result().get_counts()

print("Modified Bell state:", counts_bell)
display(plot_histogram(counts_bell, title="Modified Bell state"))


# Exercise 3.2: create a GHZ state on three qubits.
# Uncomment this section once done with 3.1.
'''
qc_ghz = QuantumCircuit(3, 3)
# Add your gates here.
qc_ghz.measure([0, 1, 2], [0, 1, 2])

# Run Exercise 3.2.
counts_ghz = simulator.run(
    transpile(qc_ghz, simulator),
    shots=1000
).result().get_counts()

print("GHZ state:", counts_ghz)
display(plot_histogram(counts_ghz, title="GHZ state"))
'''

### Exercise 3 explanation

For the first circuit, we begin by creating $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ using H followed by CNOT. Applying X to one of the qubits (doesn't matter which) then flips that qubit in both possibilities: $|00\rangle \rightarrow |01\rangle$ and $|11\rangle \rightarrow |10\rangle$. This gives $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$, so measuring the circuit produces only `01` and `10`.

For the GHZ state, H on qubit 0 first creates two possibilities: qubit 0 is 0 or 1. The first CNOT connects qubit 1 to this value, producing the two-qubit pattern $|00\rangle + |11\rangle$. A second CNOT from qubit 0 to qubit 2 extends this pattern to all three qubits, producing $(|000\rangle + |111\rangle)/\sqrt{2}$. Therefore, measuring the circuit produces only `000` and `111`.

The completed circuits and their measurement distributions are shown below.

In [ ]:
# Exercise 3.1 solution: create the |Psi+> Bell state.
qc_bell2_solution = QuantumCircuit(2, 2)
qc_bell2_solution.h(0)
qc_bell2_solution.cx(0, 1)
qc_bell2_solution.x(0)
qc_bell2_solution.measure([0, 1], [0, 1])

# Exercise 3.2 solution: create a three-qubit GHZ state.
qc_ghz_solution = QuantumCircuit(3, 3)
qc_ghz_solution.h(0)
qc_ghz_solution.cx(0, 1)
qc_ghz_solution.cx(0, 2)
qc_ghz_solution.measure([0, 1, 2], [0, 1, 2])


# Show the completed circuits.
display(qc_bell2_solution.draw("mpl"))
display(qc_ghz_solution.draw("mpl"))


# Run each circuit and show its measurement distribution.
for circuit, label in [
    (qc_bell2_solution, "Psi+ Bell state: 01 or 10"),
    (qc_ghz_solution, "GHZ state: 000 or 111")
]:
    counts = simulator.run(
        transpile(circuit, simulator),
        shots=1000
    ).result().get_counts()

    print(label + ":", counts)
    display(plot_histogram(counts, title=label))

---
## 5. The Quantum Fourier Transform, QFT


The **Quantum Fourier Transform (QFT)** is a quantum version of the discrete Fourier transform. Its purpose is to take information stored in one form and rewrite it in a form where **repeating patterns** are easier to detect.

A useful analogy is sound. A musical chord may sound like one complicated signal, but a Fourier transform can describe that same sound in terms of the individual frequencies that make it up. The information has not changed; we have changed the way we represent it.

This matters because a repeating pattern has a **period**: the distance before the pattern repeats. For example,

`1, 0, 0, 1, 0, 0, 1, 0, 0, ...` has a period of 3.

This is why the QFT matters for Shor's algorithm: the central quantum task in Shor's algorithm is finding a period. We will see below how the QFT helps make that period detectable.

### Amplitude and phase

So far, we have mostly focused on the probability of measuring a qubit as 0 or 1. However, a quantum amplitude contains more information than just its size: it also has a **phase**.

We have already seen an example:

$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}.$$

If either state is measured directly, there is a 50% chance of getting 0 and a 50% chance of getting 1. However, the states are not identical. The minus sign in $|-\rangle$ represents a difference in **relative phase**.

More generally, we can write a state such as

$$\frac{|0\rangle + e^{i\theta}|1\rangle}{\sqrt{2}}.$$

The factor $e^{i\theta}$ represents a phase rotation by an angle $\theta$. You do not need to calculate complex exponentials by hand here. The important idea is that changing phase does not necessarily change the immediate measurement probabilities, but it changes how amplitudes **interfere** later.

This is something we have already used. Applying H to $|+\rangle$ makes the amplitudes reinforce toward $|0\rangle$, while applying H to $|-\rangle$ makes them reinforce toward $|1\rangle$.

The QFT takes this idea much further: it creates carefully chosen phase relationships so that interference can reveal useful patterns in a quantum state.

### What does the QFT transform?

For $n$ qubits, there are

$$N = 2^n$$

possible computational basis states. A general quantum state can therefore have amplitudes associated with $|0\rangle, |1\rangle, \ldots, |N-1\rangle$.

The QFT transforms those amplitudes into a new set of amplitudes with carefully arranged phases. For a single basis state $|x\rangle$, the transformation is

$$QFT|x\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i xk/N}|k\rangle.$$

You do **not** need to calculate this formula manually for the circuits in this notebook. Its pieces tell us what the QFT is doing:

- $N = 2^n$ is the number of possible basis states.
- $x$ represents the input basis state.
- $k$ runs through every possible output basis state.
- $1/\sqrt{N}$ gives the terms equal magnitude.
- $e^{2\pi i xk/N}$ gives the terms different phases.

So, when applied to a computational basis state, the QFT creates an equal-magnitude superposition with a **specific pattern of phases**.

### We have already seen the smallest QFT

For a single qubit, $N=2$. The QFT transforms the two possible basis states as

$$|0\rangle \rightarrow \frac{|0\rangle+|1\rangle}{\sqrt{2}} = |+\rangle,$$

$$|1\rangle \rightarrow \frac{|0\rangle-|1\rangle}{\sqrt{2}} = |-\rangle.$$

<details>
<summary><b>Optional: see how the QFT formula gives these results</b></summary>

For one qubit, $n=1$, so the number of basis states is $N=2^n=2$. The QFT formula therefore becomes

$$QFT|x\rangle=\frac{1}{\sqrt2}\sum_{k=0}^{1}e^{2\pi i xk/2}|k\rangle.$$

The sum simply tells us to evaluate one term for $k=0$ and another for $k=1$.

For $x=0$:

$$QFT|0\rangle=\frac{1}{\sqrt2}\left(e^{2\pi i(0)(0)/2}|0\rangle+e^{2\pi i(0)(1)/2}|1\rangle\right)$$

$$=\frac{1}{\sqrt2}\left(e^0|0\rangle+e^0|1\rangle\right)$$

$$=\frac{|0\rangle+|1\rangle}{\sqrt2}.$$

For $x=1$:

$$QFT|1\rangle=\frac{1}{\sqrt2}\left(e^{2\pi i(1)(0)/2}|0\rangle+e^{2\pi i(1)(1)/2}|1\rangle\right)$$

$$=\frac{1}{\sqrt2}\left(e^0|0\rangle+e^{i\pi}|1\rangle\right)$$

Using $e^0=1$ and $e^{i\pi}=-1$,

$$=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

So the general QFT formula really does produce the two transformations shown above.

</details>

But these are exactly the transformations performed by the Hadamard gate:

$$H|0\rangle=|+\rangle, \qquad H|1\rangle=|-\rangle.$$

Therefore, for one qubit,

$$QFT_2 = H.$$

Larger QFT circuits extend this idea across multiple qubits, using Hadamard gates together with controlled phase rotations.

### What goes into a three-qubit QFT?

This section explains in detail how a three-qubit QFT works mathematically. It is not necessary to understand every calculation here to understand or implement the QFT circuit, so feel free to skip ahead to the circuit construction if you are mainly interested in the implementation.

The circuit would have three qubits, so $n=3$ and

$$N=2^3=8.$$

It can therefore take any three-qubit state as input. For a computational basis state, the possible inputs range from $|000\rangle$ to $|111\rangle$.

For example, the simplest input is $|000\rangle$. Since $x=0$, every phase in the QFT formula is $e^0=1$, giving

$$QFT|000\rangle=\frac{1}{\sqrt{8}}\left(|000\rangle+|001\rangle+|010\rangle+|011\rangle+|100\rangle+|101\rangle+|110\rangle+|111\rangle\right).$$

Every basis state has the same amplitude and phase.

For another input, such as $|001\rangle$ ($x=1$ here; $x$ is simply the decimal value represented by the input bitstring, so $|001\rangle$ corresponds to $x=1$, $|010\rangle$ to $x=2$, and so on up to $|111\rangle$, where $x=7$), the output still contains all eight basis states with equal amplitude magnitude, but now their **phases are different**:

$$QFT|001\rangle=\frac{1}{\sqrt{8}}\sum_{k=0}^{7}e^{2\pi i k/8}|k\rangle.$$

Expanding the sum gives

$$QFT|001\rangle=\frac{1}{\sqrt{8}}\left(|000\rangle+e^{i\pi/4}|001\rangle+e^{i\pi/2}|010\rangle+e^{3i\pi/4}|011\rangle-|100\rangle+e^{5i\pi/4}|101\rangle+e^{3i\pi/2}|110\rangle+e^{7i\pi/4}|111\rangle\right).$$

Notice that every term still has amplitude magnitude $1/\sqrt{8}$; what changes from term to term is its **phase**.

The phases do **not** represent how far an output state is from the original input. Instead, they are angles that determine how the amplitudes can interfere later. For $|001\rangle$, the phase increases by $\pi/4$ for each successive basis state:

$$0,\ \frac{\pi}{4},\ \frac{\pi}{2},\ \frac{3\pi}{4},\ \pi,\ \frac{5\pi}{4},\ \frac{3\pi}{2},\ \frac{7\pi}{4}.$$

Phase should be thought of as an angle around a circle, so a larger angle does not mean a state is more important or farther away. For example, $7\pi/4$ is equivalent to $-\pi/4$. The important information is the **pattern of phases across the basis states**. When the QFT acts on states containing periodic structure, these phase patterns cause some amplitudes to reinforce and others to cancel, making information about the period easier to extract.

For example, consider the three-qubit state

$$|\psi\rangle=\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle).$$

The occupied basis states are $0,2,4,6$, so they are spaced apart by a period of 2.

Consider what happens to the amplitude of the output $|001\rangle$ after the QFT. Since $|001\rangle$ corresponds to $k=1$, the four input states contribute the following phases:

$$x=0:\qquad e^{2\pi i(0)(1)/8}=e^0=1,$$

$$x=2:\qquad e^{2\pi i(2)(1)/8}=e^{i\pi/2}=i,$$

$$x=4:\qquad e^{2\pi i(4)(1)/8}=e^{i\pi}=-1,$$

$$x=6:\qquad e^{2\pi i(6)(1)/8}=e^{3i\pi/2}=-i.$$

When these contributions are added together,

$$1+i-1-i=0.$$

They therefore **cancel through destructive interference**, so the probability of measuring $|001\rangle$ is zero.

For another output, $|100\rangle$, which corresponds to $k=4$, the four input states contribute

$$x=0:\qquad e^{2\pi i(0)(4)/8}=e^0=1,$$

$$x=2:\qquad e^{2\pi i(2)(4)/8}=e^{2\pi i}=1,$$

$$x=4:\qquad e^{2\pi i(4)(4)/8}=e^{4\pi i}=1,$$

$$x=6:\qquad e^{2\pi i(6)(4)/8}=e^{6\pi i}=1.$$

When these contributions are added together,

$$1+1+1+1=4.$$

They therefore **reinforce through constructive interference**, giving that output a large amplitude.

We have only checked two of the eight possible outputs so far. If we perform the same calculation for every value of $k$, we find that the contributions cancel completely for

$$k=1, 2, 3, 5, 6, \text{and } 7$$

while they reinforce for

$$k=0\quad\text{and}\quad k=4.$$

We can therefore calculate the complete QFT of the original periodic state. Remember that the original state contributes a factor of $1/2$, while the QFT contributes a factor of $1/\sqrt{8}$. For $k=0$ and $k=4$, the four phase contributions add to 4, so the amplitude of each surviving output is

$$\frac{1}{2}\frac{1}{\sqrt{8}}(4)=\frac{1}{\sqrt{2}}.$$

All of the other output amplitudes are zero because their phase contributions cancel. Therefore,

$$QFT|\psi\rangle=\frac{1}{\sqrt{2}}\left(|000\rangle+|100\rangle\right).$$

If we now measured the state, the only possible results would be

$$|000\rangle\quad\text{or}\quad|100\rangle,$$

each with probability

$$\left|\frac{1}{\sqrt{2}}\right|^2=\frac{1}{2}.$$

This result is especially important because of where those two peaks occur. The original input had a period of

$$r=2,$$

and there are

$$N=8$$

possible basis states.

To understand what this means, we can think of the amplitudes of the original state as the pattern

$$[1,0,1,0,1,0,1,0].$$

Ignoring normalization, the pattern $[1,0]$ repeats every 2 positions, which is why its period is $r=2$. Since there are 8 positions total, this pattern repeats

$$\frac{N}{r}=\frac{8}{2}=4$$

times across the entire state.

The QFT converts this **period information** into **frequency information**. Instead of representing the fact that the pattern repeats every 2 positions, the QFT produces peaks corresponding to the fact that the pattern repeats 4 times across the 8 positions.

This is exactly what we found: the surviving QFT outputs are $k=0$ and $k=4$. The peak at $k=4$ represents the frequency of 4 repetitions across the input. A peak also appears at $k=0$ because all four input amplitudes contribute positively there, so they reinforce rather than cancel. The important peak for identifying the repeating pattern is $k=4$.
Therefore, the QFT has converted

$$\text{period }2\quad\longrightarrow\quad\text{frequency }4.$$

It is not simply preserving some of the original states and removing others. The outputs $|000\rangle$ and $|100\rangle$ survive because the phase contributions from all four input states interfere constructively at those output indices, while they interfere destructively at the others.

<details>
<summary><b>Optional: see another example with a different period</b></summary>

Consider the three-qubit state

$$|\phi\rangle=\frac{1}{\sqrt{2}}(|000\rangle+|100\rangle).$$

The occupied basis states are $0$ and $4$, so they are spaced apart by a period of 4. Ignoring normalization, its amplitude pattern is

$$[1,0,0,0,1,0,0,0].$$

The pattern $[1,0,0,0]$ repeats every 4 positions. Since there are $N=8$ positions total, it repeats

$$\frac{N}{r}=\frac{8}{4}=2$$

times. We should therefore expect the QFT to reveal a frequency of 2.

Let's calculate the QFT. For each possible output $k$, the two input states $x=0$ and $x=4$ contribute phases

$$e^{2\pi i(0)k/8}=1$$

and

$$e^{2\pi i(4)k/8}=e^{i\pi k}=(-1)^k.$$

If $k$ is even, then $(-1)^k=1$, so the contributions reinforce:

$$1+1=2.$$

If $k$ is odd, then $(-1)^k=-1$, so the contributions cancel:

$$1-1=0.$$

Therefore, the surviving outputs are

$$k=0,\quad2,\quad4,\quad6.$$

Including the normalization factors, the QFT gives

$$QFT|\phi\rangle=\frac{1}{2}\left(|000\rangle+|010\rangle+|100\rangle+|110\rangle\right).$$

Each surviving output therefore has probability

$$\left|\frac{1}{2}\right|^2=\frac{1}{4}.$$

Notice that the surviving peaks are separated by 2:

$$0,\quad2,\quad4,\quad6.$$

This matches

$$\frac{N}{r}=\frac{8}{4}=2.$$

So in this example,

$$\text{input period }4\quad\longrightarrow\quad\text{frequency }2.$$

This is the reverse of our previous example: a pattern that repeats every 2 positions produced peaks separated by 4, while a pattern that repeats every 4 positions produces peaks separated by 2.

</details>

### Why does this phase pattern matter?

For a single input such as $|001\rangle$, the QFT creates a particular phase pattern across the output basis states. The input value determines how quickly that phase changes. From

$$e^{2\pi i xk/N},$$

increasing $x$ makes the phase rotate more quickly as $k$ moves through the output states. In this sense, different input values produce different "phase frequencies."

By itself, this is not yet especially useful. The power of the QFT appears when the input is a **superposition of many basis states**. Each input basis state contributes its own phase pattern to the same output states. Those contributions then add together.

As we saw in the example above, when the input contains a periodic superposition such as

$$|\psi\rangle=\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle),$$

the occupied basis states are spaced every 2 positions. When the QFT transforms this entire superposition, the phase contributions from these four terms combine at each output state. At some outputs, their amplitudes point in different directions and cancel through **destructive interference**. At others, they line up and add through **constructive interference**.

In this example, that interference transformed

$$\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle)$$

into

$$\frac{1}{\sqrt{2}}(|000\rangle+|100\rangle).$$

Why is this useful? Suppose we did not already know that the original state had a period of 2. If we measured the original state directly, we would only obtain one of

$$0,\quad2,\quad4,\quad6.$$

A single result does not reveal that these values are spaced every 2 positions because measurement does not let us see the entire superposition at once.

After the QFT, however, the hidden periodic structure affects where the measurement probabilities are concentrated. In this example, the period of 2 produces a frequency peak at $k=4$. Since

$$\frac{N}{r}=4,$$

we can work backwards from this frequency information to obtain information about the original period.

More generally, for a perfectly periodic pattern whose period $r$ divides $N$, the QFT produces peaks separated by $N/r$. The QFT therefore gives us a way to extract information about a period that was previously encoded across many components of a superposition and could not be observed directly with a single measurement.

If $r$ does not divide $N$ exactly, these peaks instead occur near multiples of $N/r$. In Shor's algorithm, these approximate peak locations are measured and classical processing is then used to recover the period $r$.

This is why the QFT matters for Shor's algorithm: Shor's quantum circuit creates a state containing periodic structure, and the QFT converts that hidden structure into measurement outcomes from which we can infer the period.

So the three-qubit QFT does not normally turn one bitstring into another bitstring. It transforms the amplitudes of the input into a new superposition, where interference can make patterns such as periodicity detectable through measurement.

### How do gates create these phase patterns?

We now know what the QFT needs to produce: a superposition with very specific relative phases. The remaining question is how quantum gates can create it.

The gates in the QFT circuit each have a specific job:

**Hadamard gates (H)** create equal superpositions. For example,

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}.$$

This gives us the equal-magnitude amplitudes that appear in the QFT.

**Controlled-phase gates** then create the required relative phases. A controlled-phase rotation by an angle $\theta$ leaves $|00\rangle$, $|01\rangle$, and $|10\rangle$ unchanged, while

$$|11\rangle\rightarrow e^{i\theta}|11\rangle.$$

It therefore changes phase without flipping either qubit. In Qiskit, we apply one using

`circuit.cp(theta, control, target)`

The QFT uses progressively smaller phase rotations,

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

so that each qubit receives the particular phase corrections required by the QFT. The circuit repeats the same basic process across the qubits: **create a superposition with H, then add the necessary phase corrections with controlled-phase gates.**

Why do these particular angles appear? They come directly from the phase term in the QFT formula,

$$e^{2\pi i xk/N}.$$

For an $n$-qubit system, $N=2^n$. Consider our three-qubit example, where $N=8$. The phase term is therefore

$$e^{2\pi i xk/8}.$$

The three qubits represent a binary number. For example,

$$|101\rangle \rightarrow 5 = 4+1.$$

The three binary positions have values

$$4,\quad 2,\quad 1.$$

Recall that the QFT phase is

$$e^{2\pi i xk/N}.$$

The important part for us here is the fraction $xk/N$. Because $x$ is built from the values represented by its qubits, each binary place contributes part of $x$ and therefore contributes part of this fraction.

For three qubits, $N=8$. A qubit in the 4's place can contribute 4 to $x$, so its corresponding contribution inside the fraction is $4/8$. Similarly, the 2's place contributes $2/8$, and the 1's place contributes $1/8$.

When these values appear divided by $N=8$ in the QFT phase, they become

$$\frac{4}{8}=\frac{1}{2},\qquad
\frac{2}{8}=\frac{1}{4},\qquad
\frac{1}{8}=\frac{1}{8}.$$

These fractions determine how much each binary position can contribute to the phase. Each position to the right contributes half as much as the previous one.

But these fractions are not themselves angles. The QFT formula places them inside an exponential of the form

$$e^{2\pi i(\text{fraction})}.$$

Earlier, we wrote a phase as $e^{i\theta}$, where $\theta$ is the phase angle. Comparing

$$e^{2\pi i(\text{fraction})}$$

with

$$e^{i\theta},$$

we can see that the corresponding phase angle is

$$\theta=2\pi(\text{fraction}).$$

In other words, a fraction tells us what fraction of one complete $2\pi$ phase rotation is required. A contribution of $1/2$ means half of a full rotation, a contribution of $1/4$ means one quarter of a full rotation, and so on.

The first of these contributions, $1/2$, corresponds to a phase angle of

$$2\pi\left(\frac{1}{2}\right)=\pi.$$

This part is already produced by the Hadamard gate: recall that

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}},$$

while

$$H|1\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}
=\frac{|0\rangle+e^{i\pi}|1\rangle}{\sqrt{2}}.$$

So the Hadamard already accounts for the $\pi$ phase associated with the first binary contribution.

The controlled-phase gates then add the smaller phase contributions from the other qubits. The next binary contribution is $1/4$, giving

$$2\pi\left(\frac{1}{4}\right)=\frac{\pi}{2}.$$

The next is $1/8$, giving

$$2\pi\left(\frac{1}{8}\right)=\frac{\pi}{4}.$$

With more qubits, the next would be $1/16$, giving

$$2\pi\left(\frac{1}{16}\right)=\frac{\pi}{8},$$

and the pattern continues.

Therefore, the controlled-phase rotations required by the QFT are

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

Each time the control qubit is one position farther from the target qubit, its required phase correction is half as large. If the control and target are $d$ positions apart, the required angle is

$$\theta=\frac{\pi}{2^d}.$$

In the code below, the distance $d$ is `target - control`, so we calculate the angle using

`angle = pi / (2 ** (target - control))`.

In [ ]:
def build_qft(n_qubits):
    """Build a Quantum Fourier Transform (QFT) circuit on n_qubits."""

    # Create the circuit that will contain the QFT.
    circuit = QuantumCircuit(n_qubits, name="QFT")

    # Build the QFT from the highest-indexed qubit downward.
    # For 3 qubits: target = 2, then 1, then 0.
    for target in reversed(range(n_qubits)):

        # Create an equal superposition on the current target qubit.
        circuit.h(target)

        # Apply the phase corrections from each lower-indexed qubit.
        # For 3 qubits:
        #   target = 2 -> controls 0 and 1
        #   target = 1 -> control 0
        #   target = 0 -> no controls
        for control in range(target):

            # The phase rotation becomes smaller as the distance between
            # the control and target increases:
            #   adjacent qubits -> pi/2
            #   two apart       -> pi/4
            #   three apart     -> pi/8
            angle = pi / (2 ** (target - control))

            # Apply the controlled-phase rotation discussed above.
            # When both selected qubits are |1>, that component of the
            # state receives a phase factor of e^(i*angle).
            circuit.cp(angle, control, target)

    # The QFT construction leaves the qubit order reversed.
    # Swap the outer qubits to restore the conventional output order.
    # For 3 qubits, this swaps qubits 0 and 2.
    for left in range(n_qubits // 2):
        circuit.swap(left, n_qubits - left - 1)

    return circuit


# Build and display the 3-qubit QFT discussed above.
qft3 = build_qft(3)
qft3.draw("mpl")

### Reading the three-qubit QFT circuit

The circuit above is the three-qubit QFT that we just constructed in code. Reading from left to right, we can see the same pattern used repeatedly:

1. An **H gate** creates a superposition on the current target qubit.
2. **Controlled-phase gates** add the relative phases required by the QFT. The labels such as $\pi/2$ and $\pi/4$ show the angle of each phase rotation.
3. The same process is repeated for the remaining qubits.

Notice that qubits farther apart use smaller controlled-phase rotations. As we derived above, each additional position between the control and target halves the required angle:

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

At the end of the circuit, we see **SWAP gates**. These do not create the Fourier transform itself. The sequence of H and controlled-phase gates naturally produces the QFT output with the qubit positions reversed. For example, in a three-qubit circuit, the position that would normally be read first appears last and vice versa. The SWAP between qubits 0 and 2 reverses these outer positions so that the final result is displayed in the conventional qubit order. The middle qubit does not need to move.

The important idea is therefore not to memorize the circuit. The H gates create superpositions, the controlled-phase gates create the carefully chosen relative phases that produce the required interference, and the final SWAP gates put the resulting qubits in the expected order.

You can also experiment with the circuit yourself. Try changing

`qft3 = build_qft(3)`

to `build_qft(4)`, `build_qft(5)`, or another number of qubits. As the number of qubits increases, notice how the same structure continues: more H gates are required, more controlled-phase gates appear with progressively smaller angles, and the appropriate outer qubits are swapped at the end.

### QFT: main takeaways

Before moving on, here are the main ideas to remember from this section:

- **The QFT is a change of representation.** Like a classical Fourier transform, it can turn information about repeating patterns into information about frequencies.

- **Quantum amplitudes have both magnitude and phase.** Two states can have the same immediate measurement probabilities but behave differently later because their relative phases are different.

- **The QFT uses phase to create interference.** When multiple input amplitudes contribute to the same output, their phases determine whether they reinforce through constructive interference or cancel through destructive interference.

- For $n$ qubits, there are $N=2^n$ computational basis states, and the QFT of a basis state $|x\rangle$ is

  $$QFT|x\rangle=\frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}e^{2\pi i xk/N}|k\rangle.$$

  For a single basis-state input, this produces an equal-magnitude superposition whose relative phases depend on $x$.

- **The one-qubit QFT is simply the Hadamard gate.** Larger QFT circuits extend the same idea using Hadamard gates and controlled-phase rotations.

- **The QFT can reveal periodic structure hidden across a superposition.** A direct measurement only gives one outcome and cannot show us the entire pattern at once. After the QFT, interference concentrates measurement probability at particular output values related to the period.

- In the clean case where a period $r$ divides $N$, the QFT produces peaks separated by

  $$\frac{N}{r}.$$

  If $r$ does not divide $N$ exactly, the peaks instead occur near these values. This is the connection that will eventually allow us to use the QFT for period finding in Shor's algorithm.

- **The controlled-phase angles are not arbitrary.** They come from the phase $e^{2\pi i xk/N}$ in the QFT definition. Because the register represents binary values, the required phase contributions decrease by powers of two, producing rotations such as

  $$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

- **The circuit implementation reflects all of these ideas.** Hadamard gates create the required superpositions, controlled-phase gates build the required phase relationships, and SWAP gates correct the final qubit ordering.

The central idea to carry forward is:

> **The QFT uses carefully chosen phases and quantum interference to transform hidden periodic structure into measurement outcomes from which information about the period can be extracted.**

We now understand what the QFT does and how to construct it. Next, we will look at how to reverse this transformation using the **inverse Quantum Fourier Transform**.

### The inverse QFT

Quantum gates are reversible, and the QFT is no exception. The **inverse Quantum Fourier Transform**, written $QFT^\dagger$ or $QFT^{-1}$, undoes the QFT:

$$QFT^{-1}(QFT|\psi\rangle)=|\psi\rangle.$$

To reverse the transformation, the gates are effectively applied in reverse order and the phase rotations are reversed.

The symbol $\dagger$ is called a **dagger**. In linear algebra it represents the conjugate transpose of an operation, but for our purposes you can think of $QFT^\dagger$ as the operation that runs the QFT backward.

Qiskit can automatically construct the inverse of a circuit using `.inverse()`.

In [ ]:
def build_inverse_qft(n_qubits):
    """Build the inverse QFT on n_qubits."""
    return build_qft(n_qubits).inverse()


inv_qft3 = build_inverse_qft(3)
inv_qft3.draw("mpl")

### Checking that the inverse really works

We can test this directly. We will prepare a basis state $|101\rangle$, apply the QFT, then immediately apply the inverse QFT.

If the two circuits truly undo one another, measuring at the end should return the state we started with.

In [ ]:
# Create a 3-qubit circuit with 3 classical bits.
test_qft = QuantumCircuit(3, 3)

# Prepare |101> by flipping qubits 0 and 2 from |0> to |1>.
test_qft.x(0)
test_qft.x(2)

# Add our 3-qubit QFT circuit onto test_qft.
# inplace=True modifies test_qft itself instead of creating a new circuit.
test_qft.compose(build_qft(3), inplace=True)

# Add the inverse QFT immediately after the QFT.
# This should undo the transformation above.
test_qft.compose(build_inverse_qft(3), inplace=True)

# Measure each qubit into its corresponding classical bit.
test_qft.measure([0, 1, 2], [0, 1, 2])

# Run the complete circuit 1000 times and collect the measurement results.
counts = simulator.run(
    transpile(test_qft, simulator),
    shots=1000
).result().get_counts()

print(counts)
plot_histogram(counts)

The circuit began in $|101\rangle$. The QFT changed its representation, and the inverse QFT then reversed that transformation. We therefore measure `101` again.

The intermediate QFT state would generally not look like $|101\rangle$ if we measured it. The point is that the QFT preserves the quantum information while representing it differently, allowing phase and periodic structure to be manipulated before the inverse transformation is applied.

The histogram should contain only `101`, showing that the inverse QFT successfully reversed the QFT and returned the system to its original state.

There is one new Qiskit method in this example: `.compose()`. It allows us to take one quantum circuit and add the operations from another circuit to it. For example,

`test_qft.compose(build_qft(3), inplace=True)`

takes all of the gates in the circuit returned by `build_qft(3)` and adds them to `test_qft`.

The first argument, `build_qft(3)`, is therefore the circuit that we want to add. The parameter `inplace=True` tells Qiskit to modify `test_qft` directly. Without `inplace=True`, `.compose()` would instead return a new circuit containing the combined operations, and we would need to store that returned circuit ourselves.

We then use `.compose()` again to place the inverse QFT after the QFT:

`test_qft.compose(build_inverse_qft(3), inplace=True)`

The resulting circuit therefore performs

$$|101\rangle
\;\longrightarrow\;
QFT|101\rangle
\;\longrightarrow\;
QFT^{-1}(QFT|101\rangle)
\;\longrightarrow\;
|101\rangle.$$

This is why the final measurement returns `101`.

### Connecting this to Shor's algorithm

In the period-finding procedure used in Shor's algorithm, it is typically the **inverse QFT** that appears directly in the quantum circuit. It converts phase information encoded in the quantum state into measurement outcomes from which information about the period can be recovered.

The QFT and inverse QFT contain the same Fourier-transform relationship in opposite directions. Which direction is used depends on how the quantum state has been prepared and where the Fourier transform appears in the algorithm.

Before putting this into Shor's algorithm, let's practice using the QFT and inverse QFT ourselves.

### Exercise 4: QFT round trip

Prepare the three-qubit binary value 5, written `101`. Apply the QFT, then apply the inverse QFT, and finally measure the result. Use .compose() in your code as well as the functions build_qft() and build_inverse_qft().

If the inverse QFT correctly undoes the QFT, what result should you expect to measure?

In [ ]:
# Exercise 4: QFT round trip on |101>.
qc = QuantumCircuit(3, 3)

# Prepare |101>.


# Apply the QFT.


# Apply the inverse QFT.


# Measure all three qubits.


# Run the circuit.
counts = simulator.run(
    transpile(qc, simulator),
    shots=1000
).result().get_counts()

print(counts)
plot_histogram(counts, title="QFT followed by inverse QFT")

### Exercise 4 explanation

To prepare $|101\rangle$, we apply X to qubits 0 and 2, changing them from 0 to 1 while leaving qubit 1 unchanged.

We then apply the QFT, which transforms $|101\rangle$ into a superposition whose information is encoded across its amplitudes and phases. We do **not** measure here, because measurement would collapse this state before we could reverse the transformation.

Next, we apply the inverse QFT. Since the inverse QFT exactly undoes the QFT,

$$QFT^{-1}(QFT|101\rangle)=|101\rangle.$$

We can now measure the three qubits. Because the original state has been recovered, every shot on an ideal simulator should return `101`.

In [ ]:
# Exercise 4 solution: QFT round trip on |101>.
qc_solution = QuantumCircuit(3, 3)

# Prepare |101>.
qc_solution.x(0)
qc_solution.x(2)

# Apply QFT and then inverse QFT.
qc_solution.compose(build_qft(3), inplace=True)
qc_solution.compose(build_inverse_qft(3), inplace=True)

# Measure all qubits.
qc_solution.measure([0, 1, 2], [0, 1, 2])

# Show the completed circuit.
display(qc_solution.draw("mpl"))

# Run and display the result.
counts = simulator.run(
    transpile(qc_solution, simulator),
    shots=1000
).result().get_counts()

print("Expected result: 101. Actual result:", counts)
display(plot_histogram(counts, title="QFT followed by inverse QFT"))

---
## Great job! You've finished Part 1!

You started with a single qubit and worked your way through quantum gates, entanglement, interference, and the Quantum Fourier Transform. That's a lot to be proud of!

Now it's time for Part 2: **Shor's algorithm**, where all these ideas come together to factor a number. You'll build the quantum period-finding circuit and see how its measurements lead to the factors of 15. If you're up for another challenge, you can then extend it to 21.

Open [Shor's Algorithm with Qiskit](shors_algorithm_with_qiskit.ipynb) to continue. It includes its own setup cells and QFT helpers, so you can start it in a fresh notebook session.

Take a breath, keep moving!

---